In [ ]:
# Importação das bibliotecas necessárias
import random
import math
import numpy
from deap import base
from deap import creator
from deap import tools
from deap import algorithms

In [ ]:
# Configuração do Problema Bin Packing (Empacotamento)
NUM_ITENS = 50           # Quantidade de itens a serem empacotados
CAPACIDADE_BIN = 100     # Capacidade máxima de cada caixa (bin)

# Gerando pesos aleatórios para os itens (usando semente fixa para ser reprodutível)
random.seed(42)
PESOS = [random.randint(10, 50) for _ in range(NUM_ITENS)]

print("Pesos dos itens:", PESOS)

Pesos dos itens: [50, 17, 11, 27, 25, 24, 18, 16, 44, 15, 47, 37, 12, 11, 15, 23, 24, 42, 48, 11, 45, 22, 44, 36, 24, 38, 47, 27, 10, 20, 37, 31, 27, 19, 23, 31, 16, 15, 34, 16, 32, 32, 48, 26, 12, 39, 44, 17, 34, 15]


In [ ]:
# Definição da Estrutura do Algoritmo Genético
# Função objetivo: Minimização (Minimizar o número de caixas e as penalidades)
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))

# Indivíduo: Uma lista que mapeia cada item para uma caixa (o índice é o item, o valor é o ID da caixa)
creator.create("Individual", list, fitness=creator.FitnessMin)

In [ ]:
toolbox = base.Toolbox()

# Gerador de caixas: no pior caso, cada item fica em 1 caixa, então temos NUM_ITENS caixas (IDs de 0 até NUM_ITENS-1)
toolbox.register("attr_bin", random.randint, 0, NUM_ITENS - 1)

# Estrutura de Inicialização
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_bin, n=NUM_ITENS)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

In [ ]:
# Função de Avaliação (Fitness e Penalidade)
def evaluate_bin_packing(individual):
    bins = {} # Dicionário para guardar o peso total em cada caixa
    
    for item_idx, bin_id in enumerate(individual):
        peso = PESOS[item_idx]
        if bin_id not in bins:
            bins[bin_id] = 0
        bins[bin_id] += peso
        
    num_bins_usados = len(bins)
    penalidade = 0
    
    # Penaliza caixas que excederam a capacidade máxima permitida
    for bin_id, peso_total in bins.items():
        if peso_total > CAPACIDADE_BIN:
            # Penalidade proporcional ao excesso de peso (multiplicador alto para desencorajar o uso acima do limite)
            penalidade += (peso_total - CAPACIDADE_BIN) * 50
            
    # Fitness é o número de caixas + as penalidades
    fitness = num_bins_usados + penalidade
    return fitness,

toolbox.register("evaluate", evaluate_bin_packing)

In [ ]:
# Operadores Genéticos (Cruzamento, Mutação e Seleção)
# Cruzamento de 2 pontos (bom para representação em formato de lista)
toolbox.register("mate", tools.cxTwoPoint)

# Mutação: muda aleatoriamente a caixa de um item (10% de probabilidade de mudar um gene/caixa especifico)
toolbox.register("mutate", tools.mutUniformInt, low=0, up=NUM_ITENS-1, indpb=0.1)

# Seleção: Torneio com tamanho 3 (seleciona o melhor entre 3 indivíduos aleatórios)
toolbox.register("select", tools.selTournament, tournsize=3)

In [ ]:
def main():
    random.seed(64)
    
    # Tamanho da População: 300 indivíduos (mais alto comparado ao dos grafos para dar mais diversidade)
    pop = toolbox.population(n=400)
    
    # Hall of Fame para guardar a melhor solução de todas as gerações
    hof = tools.HallOfFame(1)
    
    # Estatísticas (Média, Desvio Padrão, Min, Max)
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", numpy.mean)
    stats.register("std", numpy.std)
    stats.register("min", numpy.min)
    stats.register("max", numpy.max)
    
    # Algoritmo Evolutivo (eaSimple):
    # cxpb (Crossover): 0.7 - 70% de chance de cruzar
    # mutpb (Mutação): 0.2 - 20% de chance de sofrer mutação global
    # ngen (Gerações): 100 - Mais tempo para convergência
    pop, log = algorithms.eaSimple(pop, toolbox, cxpb=0.7, mutpb=0.2, ngen=400, 
                                   stats=stats, halloffame=hof, verbose=True)
    
    return pop, log, hof

In [ ]:
if __name__ == "__main__":
  
    pop, log, hof = main()
    
    best_ind = hof[0]
    print("\n--- RESULTADO FINAL ---")
    print("Fitness do Melhor Indivíduo (Caixas + Penalidade):", best_ind.fitness.values[0])
    
    # Validar as caixas usadas e os pesos
    caixas_usadas = {}
    for i, bin_id in enumerate(best_ind):
        if bin_id not in caixas_usadas:
            caixas_usadas[bin_id] = []
        caixas_usadas[bin_id].append(PESOS[i])
        
    print(f"\nNúmero Total de Caixas Utilizadas: {len(caixas_usadas)}\n")
    
    for c_id, itens in sorted(caixas_usadas.items()):
        status = "VÁLIDA" if sum(itens) <= CAPACIDADE_BIN else "INVÁLIDA (Excedeu)"
        print(f"Caixa ID {c_id:02d}: Peso Atual = {sum(itens)} / {CAPACIDADE_BIN} [{status}] | Itens (pesos): {itens}")

gen	nevals	avg    	std    	min	max 
0  	400   	1389.57	1439.53	26 	8777
1  	320   	663.81 	896.319	27 	7128
2  	312   	581.515	908.57 	27 	5231
3  	339   	537.232	792.381	26 	4382
4  	302   	453.108	766.806	25 	4981
5  	298   	426.908	701.623	25 	4280
6  	308   	507.575	805.334	27 	4778
7  	296   	546.85 	855.103	25 	4280
8  	297   	453.767	766.921	24 	4782
9  	306   	541.822	860.555	24 	4830
10 	306   	573.285	916.501	25 	4579
11 	300   	526.648	901.784	25 	5730
12 	305   	533.898	833.717	25 	4726
13 	304   	485.325	840.192	25 	4776
14 	293   	408.925	816.029	25 	6676
15 	299   	313.855	643.207	25 	3979
16 	318   	266.625	600.009	24 	3632
17 	320   	131.947	403.012	22 	4426
18 	289   	138.838	426.137	22 	3776
19 	304   	131.965	430.217	22 	3274
20 	304   	156.998	435.033	22 	3673
21 	307   	160.537	465.222	21 	3575
22 	302   	179.452	461.46 	21 	2773
23 	309   	165.088	454.151	20 	4572
24 	299   	189.942	465    	20 	2971
25 	324   	185.225	450.842	20 	3174
26 	310   	224.45 	535.822	2

In [ ]:
# --- CÁLCULO DA EFICIÊNCIA (ACURÁCIA) ---
if __name__ == "__main__":
   
    
    soma_total_pesos = sum(PESOS)
    
    # O mínimo teórico é a soma de todos os pesos dividida pela capacidade da caixa (arredondado para cima)
    minimo_teorico_caixas = math.ceil(soma_total_pesos / CAPACIDADE_BIN)
    caixas_reais_usadas = len(caixas_usadas)
    
    # Conta se alguma caixa estourou o limite de peso
    caixas_invalidas = sum(1 for itens in caixas_usadas.values() if sum(itens) > CAPACIDADE_BIN)
    
    print("\n========================================")
    print("      ANÁLISE DE EFICIÊNCIA (ACURÁCIA)  ")
    print("========================================\n")
    print(f"Soma total de todos os pesos: {soma_total_pesos}")
    print(f"Mínimo teórico matemático de caixas: {minimo_teorico_caixas}")
    print(f"Caixas realmente utilizadas pelo algoritmo: {caixas_reais_usadas}\n")
    
    if caixas_invalidas > 0:
        print(f"❌ AVISO: A solução contém {caixas_invalidas} caixa(s) que excederam a capacidade!")
        print("Nível de Acerto (Validade): REPROVADO (Solução Inválida)")
        print("Eficiência: N/A (A solução precisa ser válida primeiro)")
    else:
        # A eficiência é a razão entre o que seria a solução perfeita (limite matemático) e o que o GA encontrou
        eficiencia = (minimo_teorico_caixas / caixas_reais_usadas) * 100
        
        print("✅ Nível de Acerto (Validade): 100% (Todas as restrições foram respeitadas)")
        print(f"🎯 Eficiência da Otimização ('Acurácia'): {eficiencia:.2f}%")
        
        if eficiencia == 100.0:
            print("\n🏆 PARABÉNS! O algoritmo atingiu o Ótimo Global (A perfeição matemática) para estes itens!")
        elif eficiencia >= 90.0:
            print("\n🔥 EXCELENTE! O algoritmo chegou extremamente perto da perfeição.")


      ANÁLISE DE EFICIÊNCIA (ACURÁCIA)  

Soma total de todos os pesos: 1378
Mínimo teórico matemático de caixas: 14
Caixas realmente utilizadas pelo algoritmo: 16

✅ Nível de Acerto (Validade): 100% (Todas as restrições foram respeitadas)
🎯 Eficiência da Otimização ('Acurácia'): 87.50%
